# 05 - Entrenamiento Baseline

Este notebook entrena el primer modelo baseline para el problema de clasificacion binaria definido en la etapa anterior. El objetivo es estimar si un caso termina siendo grave o mortal a partir de caracteristicas de la victima y del contexto del siniestro.

## Hipotesis y problema predictivo

**Hipotesis:** Las caracteristicas de la victima y del contexto del siniestro permiten anticipar si el caso terminara siendo grave o mortal.

El problema se formula como una clasificacion binaria supervisada. La variable objetivo es `es_grave_o_mortal`, donde la clase positiva representa casos graves o mortales y la clase negativa representa casos leves.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "siniestros_limpio_enriquecido.csv"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_FILE = OUTPUTS_DIR / "model_metrics.json"

TARGET = "es_grave_o_mortal"
RANDOM_STATE = 42
TEST_SIZE = 0.2

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 100)

## Carga del dataset enriquecido

Se utiliza el dataset generado por la etapa de preprocessing y feature engineering. No se lee el dato crudo ni se modifican archivos en `data/raw/`.

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"No se encontro {DATA_FILE}. Ejecute primero notebooks/02_preprocessing.ipynb."
    )

df = pd.read_csv(DATA_FILE, parse_dates=["fecha_siniestro"])

print(f"Dataset cargado desde: {DATA_FILE.resolve()}")
print(f"Shape: {df.shape}")
display(df.head())

## Definicion de X e y

Se excluyen variables que contienen el desenlace o informacion posterior al evento. En particular, `gravedad_victima`, `GRAVEdad_victima`, `es_mortal`, `es_grave_o_mortal` y cualquier variable derivada directamente del target no se usan como feature. Esto evita data leakage y fuerza al modelo a aprender a partir de variables disponibles en el contexto del siniestro.

In [ ]:
if TARGET not in df.columns:
    raise KeyError(f"No se encontro el target esperado: {TARGET}")

leakage_or_target_features = [
    "GRAVEdad_victima",
    "gravedad_victima",
    "es_mortal",
    "es_grave_o_mortal",
    "fecha_fallecimiento_victima",
]

numeric_features = [
    "anio_siniestro",
    "mes_siniestro",
    "dia_semana_siniestro",
    "trimestre_siniestro",
]

categorical_features = [
    "modo_desplazamiento_victima",
    "sexo_victima",
    "rol_victima",
    "edad_grupo",
    "vulnerabilidad_usuario",
]

numeric_features = [feature for feature in numeric_features if feature in df.columns]
categorical_features = [feature for feature in categorical_features if feature in df.columns]
selected_features = numeric_features + categorical_features

model_df = df[selected_features + [TARGET]].copy()
model_df[categorical_features] = model_df[categorical_features].fillna("SIN_DATO")

X = model_df[selected_features]
y = model_df[TARGET]

print(f"Target: {TARGET}")
print(f"Features seleccionadas: {len(selected_features)}")
display(pd.DataFrame({"feature": selected_features}))

print("Variables excluidas por target/leakage:")
display(pd.DataFrame({"variable": leakage_or_target_features, "presente_en_df": [col in df.columns for col in leakage_or_target_features]}))

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

In [ ]:
feature_types = pd.DataFrame({
    "tipo": ["numerica", "categorica"],
    "variables": [numeric_features, categorical_features],
    "cantidad": [len(numeric_features), len(categorical_features)],
})

display(feature_types)
display(X.dtypes.rename("dtype").to_frame())
display(X.isna().sum().rename("nulos").to_frame())

## Train/test split

El `train_test_split` separa una parte de los datos para entrenamiento y otra para evaluacion. Esto permite estimar el desempeno del modelo sobre observaciones no vistas durante el ajuste.

Se usa `stratify=y` porque el target esta desbalanceado: la clase grave o mortal es minoritaria. La estratificacion conserva proporciones similares de clases en train y test, haciendo que la evaluacion sea mas estable y comparable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "filas": [len(X_train), len(X_test)],
    "positivos": [int(y_train.sum()), int(y_test.sum())],
    "tasa_positiva": [float(y_train.mean()), float(y_test.mean())],
})

display(split_summary)

## Pipeline de sklearn

El pipeline encapsula el preprocesamiento y el modelo en un unico objeto reproducible. Las variables categoricas se transforman con `OneHotEncoder`, mientras que las numericas se pasan sin transformacion adicional mediante `passthrough`. Como modelo inicial se utiliza `LogisticRegression`, un baseline interpretable y apropiado para clasificacion binaria.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

model

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Modelo baseline entrenado correctamente.")

## Evaluacion del modelo

La accuracy indica la proporcion total de aciertos, pero no es suficiente cuando existe desbalance de clases: un modelo podria acertar muchos casos leves y fallar justamente los casos graves o mortales. Por eso se reportan tambien precision, recall y F1.

La precision mide que proporcion de las predicciones positivas fue correcta. El recall mide que proporcion de los casos graves o mortales reales fue detectada. El F1 resume precision y recall en una sola metrica, util cuando se necesita balancear falsos positivos y falsos negativos.

In [ ]:
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
}

conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
class_report_text = classification_report(y_test, y_pred, zero_division=0)

display(pd.DataFrame([metrics]).T.rename(columns={0: "valor"}))
print("Confusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(class_report_text)

## Visualizaciones de evaluacion

La matriz de confusion permite distinguir verdaderos negativos, falsos positivos, falsos negativos y verdaderos positivos. La distribucion de predicciones ayuda a verificar si el modelo esta prediciendo ambas clases o si colapsa hacia la clase mayoritaria.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred: leve", "Pred: grave/mortal"],
    yticklabels=["Real: leve", "Real: grave/mortal"],
    ax=ax,
)
ax.set_title("Matriz de confusion - Logistic Regression baseline")
ax.set_xlabel("Prediccion")
ax.set_ylabel("Valor real")
plt.tight_layout()
plt.show()

In [ ]:
prediction_distribution = pd.Series(y_pred, name="prediccion").value_counts().sort_index()
prediction_distribution_pct = (prediction_distribution / len(y_pred) * 100).round(2)
prediction_summary = pd.DataFrame({
    "clase": prediction_distribution.index,
    "cantidad": prediction_distribution.values,
    "porcentaje": prediction_distribution_pct.values,
})

display(prediction_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(x=y_pred, hue=y_pred, order=[0, 1], palette="Set2", legend=False, ax=ax)
ax.set_title("Distribucion de predicciones")
ax.set_xlabel("Clase predicha")
ax.set_ylabel("Cantidad")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Leve", "Grave o mortal"])
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
plt.tight_layout()
plt.show()

## Guardado de metricas

Las metricas del baseline se guardan en `outputs/model_metrics.json` para dejar trazabilidad del primer resultado entrenado y poder compararlo en futuras iteraciones.

In [ ]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

metrics_payload = {
    "model": "LogisticRegression",
    "target": TARGET,
    "hypothesis": "Las caracteristicas de la victima y del contexto del siniestro permiten anticipar si el caso terminara siendo grave o mortal.",
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "features": {
        "numeric": numeric_features,
        "categorical": categorical_features,
        "excluded_leakage_or_target": leakage_or_target_features,
    },
    "data": {
        "n_rows": int(len(df)),
        "n_features": int(len(selected_features)),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "train_positive_rate": float(y_train.mean()),
        "test_positive_rate": float(y_test.mean()),
    },
    "metrics": metrics,
    "confusion_matrix": conf_matrix.tolist(),
    "classification_report": class_report,
    "prediction_distribution": prediction_summary.to_dict(orient="records"),
}

with METRICS_FILE.open("w", encoding="utf-8") as file:
    json.dump(metrics_payload, file, ensure_ascii=False, indent=2)

print(f"Metricas guardadas en: {METRICS_FILE.resolve()}")

## Cierre

Queda entrenado un primer baseline completo con `LogisticRegression`. Este resultado no busca ser el mejor modelo posible, sino establecer una referencia reproducible para comparar proximas mejoras de feature engineering, tratamiento de faltantes, balanceo de clases, calibracion de umbral o modelos alternativos.